# Testing Knowledge Graph and LLM model

## Knowledge Graph

Rancho has ingested OpenTargets into a neo4j knowledge graph using biocypher package.

To access the graph, you'll need credentials. These are supplied via `.env` file that should have **NEO4J_URI**, **NEO4J_USERNAME** and **NEO4J_PASSWORD** in it. Example URI is "bolt+s://neo4j.myhost.com:7687"


In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")

Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Here are a couple of examples on how to run arbitrary queries on Neo4j. 

First, using **neo4j** library (pip install neo4j)


In [2]:
from neo4j import GraphDatabase

# Create a driver instance
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_username, neo4j_password))

# Define a function to run your queries
def run_query():
    with driver.session() as session:
        # Execute a Cypher query
        result = session.run("MATCH (n) RETURN n LIMIT 10")
        out = []
        # Iterate over the result and print records
        for record in result:
            out.append(record)
            print(record)
    return out

# Call the function
results = run_query()

# Close the driver when done
driver.close()

print(results)

<Record n=<Node element_id='4:2714e891-0c75-41c7-9118-5df655f3fe75:0' labels=frozenset({'Obi.Disease', 'ThingWithTaxon', 'DiseaseOrPhenotypicFeature', 'BiologicalEntity', 'Entity', 'NamedThing', 'Disease'}) properties={'licence': 'https://platform-docs.opentargets.org/licence', 'code': 'http://purl.obolibrary.org/obo/OBI_1110122', 'name': 'pathological process', 'preferred_id': 'obi', 'description': 'Abnormal, harmful processes caused by or associated with a disease', 'source': 'Open Targets', 'id': 'obi:1110122', 'version': '22.11'}>>
<Record n=<Node element_id='4:2714e891-0c75-41c7-9118-5df655f3fe75:1' labels=frozenset({'Obi.Disease', 'ThingWithTaxon', 'DiseaseOrPhenotypicFeature', 'BiologicalEntity', 'Entity', 'NamedThing', 'Disease'}) properties={'licence': 'https://platform-docs.opentargets.org/licence', 'code': 'http://purl.obolibrary.org/obo/OBI_0001621', 'name': 'longitude', 'preferred_id': 'obi', 'description': 'A measurement that is the measure of the longitude coordinate of 

Another way - py2neo library - provides a higher level interface (pip install py2neo)

In [20]:
from py2neo import Graph

# Create a Graph instance
graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)

# Run a query
results = graph.run("MATCH (n) RETURN n LIMIT 10")

# Print the results
for record in results:
    print(record)

Node('BiologicalEntity', 'Disease', 'DiseaseOrPhenotypicFeature', 'Entity', 'NamedThing', 'Obi.Disease', 'ThingWithTaxon', code='http://purl.obolibrary.org/obo/OBI_1110122', description='Abnormal, harmful processes caused by or associated with a disease', id='obi:1110122', licence='https://platform-docs.opentargets.org/licence', name='pathological process', preferred_id='obi', source='Open Targets', version='22.11')
Node('BiologicalEntity', 'Disease', 'DiseaseOrPhenotypicFeature', 'Entity', 'NamedThing', 'Obi.Disease', 'ThingWithTaxon', code='http://purl.obolibrary.org/obo/OBI_0001621', description='A measurement that is the measure of the longitude coordinate of a site.', id='obi:0001621', licence='https://platform-docs.opentargets.org/licence', name='longitude', preferred_id='obi', source='Open Targets', version='22.11')
Node('BiologicalEntity', 'Disease', 'DiseaseOrPhenotypicFeature', 'Entity', 'NamedThing', 'Obi.Disease', 'ThingWithTaxon', code='http://purl.obolibrary.org/obo/OBI_0

## LLM models

We'll use Langchain framework to invoke LLM models from OpenAI, Anthropic, Mistral and other vendors.

To work with LLMs, you need to provide API keys in .env file - **OPENAI_API_KEY**, **ANTHROPIC_API_KEY**, **MISTRAL_API_KEY**.

A few examples to highlight the capabilities of LLMs in generating Cypher queries:

In [21]:

prompt = "I have a Neo4j graph with biological data that was extracted from OpenTargets using biocypher. Generate a cypher query that would help me understand what's in the graph"


In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = "gpt-4o")
result = llm.invoke(prompt)
print(result.content)

To gain an overview of the data in your Neo4j graph extracted from OpenTargets using BioCypher, you can perform a series of Cypher queries that help you understand the structure and content of the graph. Here's a step-by-step approach:

### 1. List all Node Labels
First, you want to know what types of nodes exist in your graph. This will help you understand the different entities represented.

```cypher
CALL db.labels()
```

### 2. List all Relationship Types
Next, you'll want to know what types of relationships exist between the nodes.

```cypher
CALL db.relationshipTypes()
```

### 3. Count Nodes by Label
To understand the distribution of different node types, count them.

```cypher
MATCH (n)
RETURN labels(n) AS label, count(n) AS count
ORDER BY count DESC
```

### 4. Count Relationships by Type
Similarly, count the relationships to see their distribution.

```cypher
MATCH ()-[r]->()
RETURN type(r) AS relationshipType, count(r) AS count
ORDER BY count DESC
```

### 5. Sample Nodes an

In [23]:
# 01-preview is quite expensive. 

# temperature=1 is apparently required for this model to work

llm = ChatOpenAI(model = "o1-preview-2024-09-12", temperature = 1)
result = llm.invoke(prompt)
print(result.content)

Certainly! To help you understand the contents and structure of your Neo4j graph loaded from OpenTargets using Biocypher, you can run several Cypher queries that provide insights into the nodes, relationships, and overall schema of your graph. Below are some queries along with explanations to guide you through exploring your graph.

---

### 1. List All Node Labels

**Query:**

```cypher
MATCH (n)
RETURN DISTINCT labels(n) AS NodeLabels;
```

**Explanation:**

- This query retrieves all distinct labels assigned to nodes in your graph.
- It helps you understand the different types of entities present (e.g., genes, diseases, compounds).

---

### 2. Count Nodes per Label

**Query:**

```cypher
MATCH (n)
WITH labels(n) AS lbls
UNWIND lbls AS label
RETURN label, COUNT(*) AS NodeCount
ORDER BY NodeCount DESC;
```

**Explanation:**

- This query counts the number of nodes for each label.
- `UNWIND` is used to handle nodes with multiple labels.
- It provides insight into the distribution of e

In [24]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model = "claude-3-5-sonnet-20240620")
result = llm.invoke(prompt)
print(result.content)

Certainly! To help you understand what's in your Neo4j graph containing biological data extracted from OpenTargets using BioCypher, I can suggest a series of Cypher queries that will give you an overview of the graph structure and content. Here are some queries you can try:

1. Count the number of nodes by label:

```cypher
CALL db.labels() YIELD label
CALL apoc.cypher.run("MATCH (n:`" + label + "`) RETURN count(n) AS count", {}) YIELD value
RETURN label, value.count AS count
ORDER BY count DESC
```

2. Count the number of relationships by type:

```cypher
CALL db.relationshipTypes() YIELD relationshipType
CALL apoc.cypher.run("MATCH ()-[r:`" + relationshipType + "`]->() RETURN count(r) AS count", {}) YIELD value
RETURN relationshipType, value.count AS count
ORDER BY count DESC
```

3. Get a sample of nodes for each label (adjust LIMIT as needed):

```cypher
CALL db.labels() YIELD label
CALL apoc.cypher.run("MATCH (n:`" + label + "`) RETURN n LIMIT 5", {}) YIELD value
RETURN label, col

In [25]:
from langchain_mistralai import ChatMistralAI

# open-mistral-7b is their "legacy" model. It could be deployed locally, but we'll use API to query

llm = ChatMistralAI(model = "open-mistral-7b")
result = llm.invoke(prompt)
print(result.content)

To help you understand the structure and content of your Neo4j graph, I've prepared a few Cypher queries that you can use as a starting point. These queries will help you explore different aspects of your biological data graph.

1. Get a list of all nodes and their labels:
```cypher
MATCH (n) RETURN DISTINCT n.label AS label, COUNT(n) AS node_count
```

2. Get a list of all relationships and their types:
```cypher
MATCH (n)-[r]->(m) RETURN DISTINCT r.type AS relationship_type, COUNT(r) AS relationship_count
```

3. Get a list of all nodes and their properties:
```cypher
MATCH (n) RETURN n, COLLECT(n.*) AS properties
```

4. Get a list of all relationships and their properties:
```cypher
MATCH (n)-[r]->(m) RETURN r, COLLECT(r.*) AS properties
```

5. Get a list of all drugs and their targets:
```cypher
MATCH (d:Drug)<-[:HAS_TARGET]-(t:Target) RETURN d.name AS drug_name, t.name AS target_name
```

6. Get a list of all drugs and their side effects:
```cypher
MATCH (d:Drug)<-[:HAS_SIDE_EFF

In [5]:
from langchain.chains import GraphCypherQAChain
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI


graph = Neo4jGraph(url=neo4j_uri, username=neo4j_username, password=neo4j_password)